In [ ]:
# Purpose - computes plasmoid fraction 

findex = 140
#old 67 #100 #9 #63 #24 alte #63 #113 #121 #141 #130 #121
#46 #67 #100 #9 #63 #24
#24 done, 67 done, 9 done, 100 done, 24 done 

step_choice = "penultimate" #last for 100, rest was penultimate
#did 100, 9, 67, 24, 140 all done

In [ ]:
# Imports and Folder Selection

import skimage
from skimage import measure

from graphet import Data
from graphet.plugins import TristanV2 
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
import sys; sys.path.append('..')
from useful_funcs import *
sys.path.append('../fmgmt')
from TristanRun import TristanRun

plt.close('all')

%matplotlib widget

set_standard_plot_params()

# Read Input File Data

run = TristanRun(findex)
gyrofreq = run.gyrofreq

cc = run.cc
B0 = run.B0

tstep_choices = {"first": [run.steps[0]], "last": [run.steps[-1]], "all": run.steps, "penultimate": [run.steps[-2]]}

my_steps = tstep_choices[step_choice]
boilerplate = run.boilerplate + f" tstep = {my_steps[0]}"

In [ ]:
# Configure Data

d = run.config_data(my_steps, "fields", includes_pressure = False)

In [ ]:
# Find Maxima


TOLERANCE = 20 #20
INJ_THRESH = 0.15

delta = 80
rect = [run.xlim / 2 - delta, run.xlim / 2 + delta, 0, 500]
wholebox = [0, run.xlim, 0, run.ylim]
rect = wholebox

roi = fq_cut(d.fields, 0, rect)

roi['injected'] = (roi.epsilon - 1) / run.sigma
Az = compute_Az(roi)

allowed_xs = [run.xlim / 2 - TOLERANCE, run.xlim / 2 + TOLERANCE]

# Identify local maxima in Az
local_maxima = (Az == Az.rolling(x=3, y=3, center=True).max())

# Get the coordinates of the local maxima
local_maxima_coords = np.argwhere(local_maxima.values)

# Filter local maxima within the allowed x values
filtered_maxima_coords = np.array([coord for coord in local_maxima_coords if allowed_xs[0] <= Az.x.values[coord[1]] <= allowed_xs[1]])

# Further filter maxima to ensure roi.dens at that position is at least 5 TODO shortcut
filtered_maxima_coords = np.array([
    coord for coord in filtered_maxima_coords
    if roi.dens.values[coord[0], coord[1]] >= 5
])


In [ ]:
# Plot local maxima
plt.figure()
plot_fq3(roi, roi.dens, vabsmax = 5, positive = True)
plt.contour(roi.x, roi.y, (roi.epsilon - 1) / run.sigma, [0.15], colors = 'white')
plt.plot(Az.x.values[filtered_maxima_coords[:, 1]], Az.y.values[filtered_maxima_coords[:, 0]], 'rx')

maxima_ys = Az.y.values[filtered_maxima_coords[:, 0]]

In [ ]:
print(maxima_ys)

In [ ]:
mask = (maxima_ys > 335) & (maxima_ys < 337)

first_nonzero = np.argmax(mask)
print(first_nonzero)

In [ ]:
from matplotlib.path import Path

BOX_LEN = {0.3: 20, 10: run.xlim / 2}[run.sigma]

BS_LIM = 10

# Get Contours

def is_closed_contour(contour):
    return np.all(contour[0] == contour[-1])

def encloses_point(contour, point):
    path = Path(contour)
    return path.contains_point((point[0], point[1]))

def first_greater_index(arr, value):
    mask = arr.data > value
    return np.nonzero(mask)[0][0]

contours = []
progress = 1


for maximum in filtered_maxima_coords:
    [max_y, max_x] = maximum

    center_x = Az.x.values[max_x]
    center_y = Az.y.values[max_y]

    # Define the box around the maximum
    box_xmin = center_x - BOX_LEN / 2
    box_xmax = center_x + BOX_LEN / 2
    box_ymin = center_y - BOX_LEN / 2
    box_ymax = center_y + BOX_LEN / 2

    plt.gca().add_patch(
        plt.Rectangle(
            (box_xmin, box_ymin),
            BOX_LEN,
            BOX_LEN,
            linewidth=2,
            edgecolor='red',
            facecolor='none'
        )
    )

    lc_x = first_greater_index(Az.x, box_xmin)
    lc_y = first_greater_index(Az.y, box_ymin)

    max_x -= lc_x
    max_y -= lc_y

    curr_box = d.fields.sel(x = slice(box_xmin, box_xmax)).sel(y = slice(box_ymin, box_ymax)).isel(z = 0).isel(t = 0)

    local_Az = compute_Az(curr_box)

    print(f"{progress} / {len(filtered_maxima_coords)}")
    progress += 1

    run_no = 0
    left = local_Az.min()
    #right = local_Az.max()
    right = local_Az[max_y, max_x]
    best_contour = None

    while (run_no < BS_LIM):
        plevel = (left + right) / 2

        #print(f"plevel tried = {plevel.compute().item()}")

        contours_at_level = measure.find_contours(local_Az.values, level=plevel)
        enclosed_maximal_contours = [contour for contour in contours_at_level if \
                            is_closed_contour(contour) and encloses_point(contour, [max_y, max_x])]
        ncontours = len(enclosed_maximal_contours)

        if (ncontours > 1):
            raise ValueError("Too many enclosing contours found")
        
        if (ncontours == 0):
            left = plevel
        else:
            best_contour = enclosed_maximal_contours[0]
            right = plevel 

        run_no += 1

    if (best_contour is not None):
        shifted_contour = best_contour + [lc_y, lc_x]
        contours += [shifted_contour]

for contour in contours:
    plt.plot(Az.x.values[contour[:, 1].astype(int)], Az.y.values[contour[:, 0].astype(int)], color='blue')



plt.title(run.boilerplate + f"\n t = {run.steps[0]} $\sigma =$ {run.sigma}")
plt.show()


len_contours = len(contours)

if (len_contours == 0):
    raise Exception("No contours found.")
else:
    print(f"Found {len_contours} contours.")

In [ ]:
from matplotlib.path import Path
from scipy.interpolate import griddata

plt.figure()

roi['edens'] = roi.dens * (roi.epsilon - 1)

plot_fq3(roi, roi.edens, vabsmax = 5)

for contour in contours:
    plt.plot(Az.x.values[contour[:, 1].astype(int)], Az.y.values[contour[:, 0].astype(int)], color='blue')

#print(f"Total sum of edens over the whole domain: {total_edens_sum.values}")

def find_contour_mask(contour, data_array):
    spacing = data_array.x[1].values - data_array.x[0].values

    path = Path(contour)
    plt.plot(contour[:, 1] * spacing, contour[:, 0] * spacing, color='purple')

    x, y = np.meshgrid(np.arange(data_array.shape[1]), np.arange(data_array.shape[0]))
    points = np.vstack((y.ravel(), x.ravel())).T
    mask = path.contains_points(points).reshape(data_array.shape)

    print(f"Number of points contained in the mask: {mask.sum()}")
    plt.scatter(x[mask] * spacing, y[mask] * spacing, color='red', alpha=0.9)
    return mask

total_mask = None

for contour in contours:
    #print("considered!")

    new_mask = find_contour_mask(contour, roi.edens)
    if (total_mask is None):
        total_mask = new_mask
    else:
        total_mask = total_mask | new_mask

injected_mask = roi.injected > INJ_THRESH
total_mask = total_mask & injected_mask

plt.title(run.boilerplate + f"\n $\sigma =$ {run.sigma}")

plasmoid_sum = roi.edens.where(total_mask).sum().compute()
total_sum = roi.edens.where(injected_mask).sum().compute()

pfrac = plasmoid_sum.values / total_sum.values

print(f"Total sum of edens within all contours: {plasmoid_sum.values}")
print(f"Total sum of all: {total_sum.values}")
print(f"Fraction in Plasmoids: {pfrac}")

#24 per cent in run 24, 93 percent in run 63, 10 [12] percent in 9, 90 [92] percent in 100, 0.00235 in 14

#7/10 before Az max fix: 24% in 24, 92% in 67, 12% in 9

In [ ]:
file_path = "pfracs.npy"

if os.path.exists(file_path):
    loaded_data = np.load(file_path, allow_pickle=True).item()
else:
    loaded_data = {}

print("BEFORE")
print(loaded_data)

loaded_data[(run.sigma, run.ylim)] = pfrac

print("AFTER")
print(loaded_data)


np.save(file_path, loaded_data)

print(f"Successfully added/updated ylim = {run.ylim}, pfrac = {pfrac}")

In [ ]:
print(0 / 0)

plt.figure()

def compute_sum_within_contour(contour, data_array):
    path = Path(contour)
    plt.plot(contour[:, 1], contour[:, 0], color='purple')

    if not is_closed_contour(contour):
        raise ValueError("The contour is not closed")

    x, y = np.meshgrid(np.arange(data_array.shape[1]), np.arange(data_array.shape[0]))
    points = np.vstack((y.ravel(), x.ravel())).T
    mask = path.contains_points(points).reshape(data_array.shape)

    print(f"Number of points contained in the mask: {mask.sum()}")
    plt.scatter(x[mask], y[mask], color='red', alpha=0.9)
    return data_array.where(mask).sum().compute()

compute_sum_within_contour(contours[0], roi.edens)

plt.xlim([0, roi.edens.shape[1]])
plt.ylim([0, roi.edens.shape[0]])

In [ ]:
print(contours)

In [ ]:
for level in np.linspace(0.2, Az.max(), N_pain):
    contours += measure.find_contours(Az.values, level=level)

closed_contours = []
for contour in contours:
    if np.all(contour[:, 0] > 0) and np.all(contour[:, 0] < Az.shape[0] - 1) and np.all(contour[:, 1] > 0) and np.all(contour[:, 1] < Az.shape[1] - 1):
        if np.all(contour[0] == contour[-1]):
            closed_contours.append(contour)

for contour in closed_contours:
    plt.plot(roi.x.values[contour[:, 1].astype(int)], roi.y.values[contour[:, 0].astype(int)], color='red')

In [ ]:
from matplotlib.path import Path

# Plot Regions

delta = 80
rect = [run.xlim / 2 - delta, run.xlim / 2 + delta, 0, 500]
wholebox = [0, run.xlim, 0, run.ylim]

rect = wholebox

roi = fq_cut(d.fields, 0, wholebox)

ez_treal = roi.bx * roi.vely - roi.by * roi.velx 
ez_nontreal = roi.ez - ez_treal 
ez_tot = roi.ez
JdotE = roi.ex * roi.jx + roi.ey * roi.jy + roi.ez * roi.jz

Az = compute_Az(roi)

roi['ez_nonideal'] = roi.ez + roi.velx * roi.by - roi.vely * roi.bx
roi['ez_ideal'] = roi.ez - roi.ez_nonideal

dens = roi.dens1 + roi.dens2

esq, bsq, chi = sqrd(roi) 

adj_factor = run.beta_A

bsq_adj =  adj_factor ** 2 * bsq
chi_adj = (bsq_adj - esq) / (bsq_adj + esq)

plt.figure()

#Ainc = (0.16 * run.beta_A * run.cc) * run.B0 * run.output_interval

#plt.contour(cs.x, cs.y, Az, [1, 1 + Ainc], colors = 'green')

allowed_xs = [run.xlim / 2 - 40, run.xlim / 2 + 40]

plot_fq3(roi, Az)

# Identify local maxima in Az
local_maxima = (Az == Az.rolling(x=3, y=3, center=True).max())

# Get the coordinates of the local maxima
local_maxima_coords = np.argwhere(local_maxima.values)

# Filter local maxima within the allowed x values
filtered_maxima_coords = np.array([coord for coord in local_maxima_coords if allowed_xs[0] <= Az.x.values[coord[1]] <= allowed_xs[1]])

# Function to find the largest closed contour containing a given maxima
def find_largest_contour(maxima_coord, contours):
    def is_inside_contour(contour, point):
        return np.all(contour[:, 0] > 0) and np.all(contour[:, 0] < Az.shape[0] - 1) and np.all(contour[:, 1] > 0) and np.all(contour[:, 1] < Az.shape[1] - 1) and np.all(contour[0] == contour[-1])

    def encloses_point(contour, point):
        path = Path(contour)
        return path.contains_point((point[1], point[0]))

    left, right = 0, len(contours) - 1
    best_contour = None

    while left <= right:
        print(left)
        print(right)
        mid = (left + right) // 2
        contour = contours[mid]
        if is_inside_contour(contour, maxima_coord) and encloses_point(contour, maxima_coord):
            best_contour = contour
            left = mid + 1
        else:
            right = mid - 1

    return best_contour

# Plot contours for each maxima
for maxima_coord in filtered_maxima_coords:
    largest_contour = find_largest_contour(maxima_coord, contours)
    if largest_contour is not None:
        plt.plot(Az.x.values[largest_contour[:, 1].astype(int)], Az.y.values[largest_contour[:, 0].astype(int)], color='blue')
    else:
        print("NOT working!")


# Plot local maxima
plt.plot(Az.x.values[filtered_maxima_coords[:, 1]], Az.y.values[filtered_maxima_coords[:, 0]], 'rx')

plt.title(run.boilerplate + f"\n t = {run.steps[0]} $\sigma =$ {run.sigma}")
plt.xlim(rect[:2])
plt.ylim(rect[2:])
plt.show()

In [ ]:
print(0 / 0)

#roi = fq_cut(d.fields, 0, [80, 160, 0, 200])
roi = fq_cut(d.fields, 0, [0, run.xlim, 0, run.ylim])

#roi = fq_cut(d.fields, 0, [55, 65, 110, 130])

# Create a subplot with one plot
fig, ax = plt.subplots()

plt.title(f"{run.boilerplate}")
plot_basic(ax, run, roi, sdens = 0.5, cutoff = 5)

# Show the plot
plt.show()

In [ ]:
# Plot Regions

delta = 80
rect = [run.xlim / 2 - delta, run.xlim / 2 + delta, 0, 500]
wholebox = [0, run.xlim, 0, run.ylim]

rect = wholebox

roi = fq_cut(d.fields, 0, wholebox)

ez_treal = roi.bx * roi.vely - roi.by * roi.velx 
ez_nontreal = roi.ez - ez_treal 
ez_tot = roi.ez
JdotE = roi.ex * roi.jx + roi.ey * roi.jy + roi.ez * roi.jz

Az = compute_Az(roi)

roi['ez_nonideal'] = roi.ez + roi.velx * roi.by - roi.vely * roi.bx
roi['ez_ideal'] = roi.ez - roi.ez_nonideal

dens = roi.dens1 + roi.dens2

esq, bsq, chi = sqrd(roi) 

adj_factor = run.beta_A

bsq_adj =  adj_factor ** 2 * bsq
chi_adj = (bsq_adj - esq) / (bsq_adj + esq)

v_A = run.beta_A

plt.figure()

# --------------STREAMLINES--------------

#plot_streamline(plt, cs, cs.bx, cs.by, 2, 'white') 
#plot_streamline(plt, cs, cs.velx, cs.vely, 2, 'black') 

# --------------CONTOURS--------------

plt.contour(roi.x, roi.y, chi_adj, [0], colors = 'green')

Ainc = (0.16 * run.beta_A * run.cc) * run.B0 * run.output_interval

#plt.contour(cs.x, cs.y, Az, [1, 1 + Ainc], colors = 'green')

# --------------AREA PLOTS--------------
# SCALARS

#plot_fq3(cs, cs.dens, vabsmax = 3)
#plot_fq3(cs, cs.epsilon - 1, vabsmax = 2 * run.sigma, cmap = 'seismic')

# FIELDS

#plot_fq3(cs, cs.ex / B0, "$E_x / B_0$", vabsmax = 0.15, cmap = 'seismic')
#plot_fq3(cs, cs.ey / B0, "$E_y / B_0$", vabsmax = 0.3)
#plot_fq3(cs, cs.ez / B0, "$E_z / B_0$", vabsmax = 0.15)

#plot_fq3(cs, cs.by, vabsmax = 0.15)
#plot_fq3(cs, Az)

# NONIDEAL EFFECTS

#plot_fq3(cs, cs.ez / run.B0, vabsmax = 0.15)
#plot_fq3(cs, cs.ez_ideal / run.B0, cmap = "seismic", vabsmax = 0.1)
#plot_fq3(cs, cs.ez_nonideal / run.B0, cmap = "seismic", vabsmax = 0.1)
#plot_fq3(cs, cs.jz, f"$J_z$", vabsmax = 1e-4, cmap = 'seismic')
#plot_fq3(cs, cs.jz * cs.ez, "JzEz", vabsmax = 1e-7)
#plot_fq3(cs, cs.ez_nonideal / cs.jz, vabsmax = 20)

# VELOCITY

#plot_fq3(cs, cs.velx / v_A, boilerplate + "$v_x / v_A$", vabsmax = 0.6)
plot_fq3(roi, roi.vely / v_A, vabsmax = 1.5, cmap = 'seismic')

# --------------PLASMOID IDENTIFIERS--------------

# Find points where ystrip.bx switches sign
sign_change_indices = np.where(np.diff(np.sign(ystrip.bx)) > 0)[0]

plt.title(run.boilerplate + f"\n t = {run.steps[0]} $\sigma =$ {run.sigma}")
plt.vlines(ycut_x, rect[2], rect[3], colors = 'yellow', linestyle = '--')
plt.hlines(xcut_y, rect[0], rect[1], colors = 'yellow', linestyle = '--')
plt.xlim(rect[:2])
plt.ylim(rect[2:])
plt.show()

In [ ]:
print(sign_change_indices)

In [ ]:
# X-strip 

strip = xstrip

ez_ideal = strip.bx * strip.vely - strip.by * strip.velx 
ez_nonideal = strip.ez - ez_treal 
ez_tot = strip.ez
dens = strip.dens1 + strip.dens2

JdotE = strip.jx * strip.ex + strip.jy * strip.ey + strip.jz * strip.ez

esq, bsq, chi = sqrd(strip)


jadj = 200

plt.figure()
#plt.title(boilerplate + f"t = {tchosen}, y = {xcut_y}")

#plt.plot(strip.x, ez_nontreal / (cc * B0), label = "$E_{z, ni} / c B_0$", color = '#d62728')
#plt.plot(strip.x, ez_treal / (cc * B0), label = "ez_treal / c B_0")
#plt.plot(strip.x, strip.ez / (cc * B0), label = "$E_z / c B_0$", color = '#1f77b4') #ez
#plt.plot(strip.x, bsq / B0 ** 2, label = "bsq / B0^2") #bsq
#plt.plot(strip.x, strip.jz / B0 * run.c_omp, label = '$J_z (c / \omega_p) / B_0$', color = '#2ca02c') #jz

plt.plot(strip.x, strip.by / B0, label = "$B_y / B_0$", color = '#ff7f0e')
plt.plot(strip.x, strip.epsilon, label = "$\epsilon$")
#plt.plot(strip.x, np.sqrt(bsq) / B0, label = "$B / B_0$", color = '#ff7f0e')
#plt.plot(strip.x, bsq / B0 ** 2, label = "$B^2 / B_0^2$", color = '#ff7f0e')
#plt.plot(strip.x, strip.bz / B0, label = "Bz / B0")
#plt.plot(strip.x, chi, label = '$\chi$')
#plt.plot(strip.x, chi_adj, label = '$\chi_{adj}$')
#plt.plot(strip.x, strip.velx / v_A, label = "$v_x / v_A$", color = '#9467bd')
#plt.plot(strip.x, strip.vely / v_A, label = "$v_y / v_A$", color = 'magenta')
#plt.plot(strip.x, np.sqrt(strip.velx ** 2 + strip.vely ** 2) / v_A, label = "$v / v_A$", color = '#9467bd')
plt.axhline(y = 0, color = 'black', linestyle = '--')
plt.axvline(x = ycut_x, color = 'black', linestyle = '--')

#plt.plot(strip.x, strip.pb, label = "$p_B$")
#plt.plot(strip.x, strip.px, label = "$p_x$")

#plt.plot(strip.x, strip.pb + strip.px, label = "total pressure in x dir")

nbx = strip.bx / np.sqrt(bsq)
nby = strip.by / np.sqrt(bsq)
nbz = strip.bz / np.sqrt(bsq)
#ppara = strip.px * nbx + strip.py * nbx + strip.pz * nbz

#e_cr = (5 * strip.p + 2) / (5 * strip.p + 5)
#epsilon = 1 - 0.5 * (2 * ppara - strip.p)

#plt.plot(strip.x, e_cr, label = "$e_{cr}$")
#plt.plot(strip.x, epsilon, label = "$\epsilon$")

#plt.plot(strip.x, strip.px / strip.pb, label = "$beta$")

# New
#plt.plot(strip.x, strip.ex / (cc * B0), label = "$E_x / c B_0$")
#plt.plot(strip.x, strip.velz / v_A, label = "velz / v_A")
#plt.plot(strip.x, -strip.by * strip.velz / (cc * B0), label = "E_xni")
#plt.plot(strip.x, strip.dens1 / 16, label = 'n(e-)')
#plt.plot(strip.x, strip.dens2 / 16, label = 'n(e+)')
#plt.plot(strip.x, (strip.dens1 + strip.dens2)/ 16, label = '$n$')
#plt.plot(strip.x, (strip.dens1 + strip.dens2)/ 16 * (np.sqrt(bsq) / B0), label = '$(n / n_0) * (B / B_0)$', color = 'grey')
#plt.plot(strip.x, jadj * strip.jz / (dens * B0) * 16, label = f'{jadj} * j / (n / n0)')

#plt.plot(strip.x, (strip.dens2 - strip.dens1) / 16, label = 'rho')
#plt.plot(strip.x, jadj * strip.jx / B0, label = f'{jadj} * jx / B0')

#plt.plot(strip.x, ez_nonideal / strip.jz * 1/10, label = "eta_eff")
#plt.title(f"$\sigma =$ {run.sigma}")
plt.title(boilerplate + f"x strip y = {xcut_y}")
plt.xlabel('$x$ [$c/\omega_{p}$]')
plt.ylabel('Quantity')
plt.legend(loc='upper left')

plt.show()

In [ ]:
que = M_SM.values

In [ ]:
plt.figure()
plt.plot(strip.x, M_SM, label="M_SM")
plt.axhline(y=0, color='black', linestyle='--')
plt.axvline(x=ycut_x, color='black', linestyle='--')
plt.title(boilerplate + f" x strip y = {xcut_y}")
plt.xlabel('$x$ [$c/\\omega_{p}$]')
plt.ylabel('M_SM')
plt.legend(loc='center left')
plt.show()

In [ ]:
# Check if any element in in_sqrt is negative
negative_indexer = (in_sqrt < 0).compute()
negative_elements = in_sqrt[negative_indexer]

if negative_elements.size > 0:
    print("There are negative elements in in_sqrt.")
else:
    print("There are no negative elements in in_sqrt.")

In [ ]:
from scipy.integrate import simps

# Define the interval for integration
x_start, x_end = 332, 418
x_start, x_end = 340, 410


# Extract the relevant portion of the data
interval = (strip.x >= x_start) & (strip.x <= x_end)
x_values = strip.x[interval]
bsq_values = bsq[interval] / B0 ** 2

# Perform the integration
average_result = simps(bsq_values, x_values) / (x_end - x_start)

print(f"Integral of B^2 / B_0^2 over the interval [{x_start}, {x_end}] is {average_result}")

In [ ]:
print(list(strip.data_vars))

In [ ]:
# Y-strip 

strip = ystrip

strip['ez_ideal'] = strip.bx * strip.vely - strip.by * strip.velx 
strip['ez_nonideal'] = strip.ez - strip.ez_ideal 
ez_tot = strip.ez

JdotE = strip.jx * strip.ex + strip.jy * strip.ey + strip.jz * strip.ez
dens = strip.dens1 + strip.dens2

esq, bsq, chi = sqrd(strip)

adj_factor = run.beta_A ** 2


bsq_adj = adj_factor ** 2 * bsq
chi_adj = (bsq_adj - esq) / (bsq_adj + esq)

sign_change = np.diff(np.sign(chi_adj))
#y_gate_index = np.where(sign_change > 0)[-1][-1]

jadj = 200

plt.figure()
plt.title(boilerplate + f"\n t = {tchosen}, x = {ycut_x}")
#plt.plot(strip.y, ez_nontreal / (cc * B0), label = "ez_nontreal / c B_0")
#plt.plot(strip.y, ez_treal / (cc * B0), label = "ez_treal / c B_0")
#plt.plot(strip.y, strip.ez / (cc * B0), label = "$E_z / c B_0$", color = '#1f77b4') #ez
#plt.plot(strip.y, bsq / B0 ** 2, label = "bsq / B0^2") #bsq
#plt.plot(strip.y, strip.jz / B0 * run.c_omp, label = '$J_z (c / \omega_p) / B_0$', color = '#2ca02c') #jz
#plt.plot(strip.y, strip.bx / B0, label = "$B_x / B_0$", color = '#ff7f0e')
#plt.plot(strip.y, strip.bz / B0, label = "Bz / B0")
#plt.plot(strip.y, chi, label = '$\chi$')
#plt.plot(strip.y, chi_adj, label = '$\chi_{adj}$', color = 'orange')
plt.plot(strip.y, strip.vely / run.beta_A, label = "$v_y / v_A$", color = '#9467bd')
#plt.plot(strip.y, strip.velx / run.beta_A, label = "$v_y / v_A$", color = 'red')
#plt.plot(strip.y, strip.dens / 16, label = '$n$')

#plt.plot(strip.y, strip.ez_nonideal / strip.jz, label = r"$\eta$", color = 'orange')


avg_velx_v_A = np.mean(strip.velx.values / run.beta_A)
print(avg_velx_v_A)

avg_velx_v_A = np.mean(dens.values / 16)
print(avg_velx_v_A)

plt.axhline(y = 0, color = 'black', linestyle = '--')
plt.axvline(x = xcut_y, color = 'black', linestyle = '--')

#plt.plot(strip.y, jadj * ez_nontreal * strip.jz / (cc * B0 ** 2), label = f"{jadj} * ezni jz")
#plt.plot(strip.y, ez_nontreal / (cc * B0), label = "$E_{zni}$ / $c B_0$", color = '#d62728')
#plt.axvline(x = strip.y[y_gate_index + 1], color = 'orange', linestyle = '--')

#plt.plot(strip.x, ez_nontreal / strip.jz * 1/10, label = "eta_eff")
#plt.title(f"$\sigma =$ {run.sigma}")
plt.title(boilerplate + f"\n y strip x = {ycut_x}")
plt.legend(loc = 'lower right')
plt.xlabel('$y$ [$c/\omega_{p}$]')
plt.ylabel('Quantity')
#plt.ylim([-1.5, 5])
plt.show()

In [ ]:
print(y_gate[-1][-1])

In [ ]:
p